In [1]:
%pwd

'd:\\Projects\\1-Practice and Learning\\0 - COMPLETE NEW LEARNING ML and DL\\3-PROJECTS\\Deep Learning Projects\\Chest Cancer Classification End to End Project\\research'

In [2]:
# we want to go to root directory
import os

os.chdir("../")

In [3]:
%pwd

'd:\\Projects\\1-Practice and Learning\\0 - COMPLETE NEW LEARNING ML and DL\\3-PROJECTS\\Deep Learning Projects\\Chest Cancer Classification End to End Project'

**ML flow URI**

We will be connecting with dagshub which is free

In [8]:
from dotenv import load_dotenv

# load_dotenv("../.env")  # notebook is inside research/

load_dotenv()

tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
tracking_username = os.getenv("MLFLOW_TRACKING_USERNAME")
tracking_password = os.getenv("MLFLOW_TRACKING_PASSWORD")


**Load model**

In [9]:
import tensorflow as tf

model = tf.keras.models.load_model("artifacts/training/model.h5")

**Entity**

In [13]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

**Config manager**

In [14]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import create_directories, read_yaml, save_json


In [15]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/CT Scan Dataset for Project/test",
            mlflow_uri= tracking_uri ,
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

**STEP 6 : CREATE COMPONENT**

In [22]:
from pathlib import Path
from urllib.parse import urlparse

import mlflow
import tensorflow as tf
from mlflow.keras import log_model as log_keras_model


In [25]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            preprocessing_function=tf.keras.applications.vgg16.preprocess_input
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            shuffle=False,
            **dataflow_kwargs
        )


    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": float(self.score[0]), "accuracy": float(self.score[1])}
        save_json(path=Path("scores.json"), data=scores)

    
    def log_into_mlflow(self):
        mlflow.set_tracking_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            mlflow.log_params(dict(self.config.all_params))
            mlflow.log_metrics(
                {"loss": float(self.score[0]), "accuracy": float(self.score[1])}
            )
            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                log_keras_model(
                    model=self.model,
                    name="model",
                    registered_model_name="VGG16Model",
                )
            else:
                log_keras_model(model=self.model, name="model")

**STEP 7: PIPELINE**

In [26]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
   raise e

Found 369 images belonging to 2 classes.
24/24 ━━━━━━━━━━━━━━━━━━━━ 54s 2s/step - accuracy: 0.9593 - loss: 0.4157


2026/09/22 17:03:15 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in d:\Projects\1-Practice and Learning\0 - COMPLETE NEW LEARNING ML and DL\3-PROJECTS\Deep Learning Projects\Chest Cancer Classification End to End Project
2026/09/22 17:03:18 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
2026/09/22 17:03:18 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in d:\Projects\1-Practice and Learning\0 - COMPLETE NEW LEARNING ML and DL\3-PROJECTS\Deep Learning Projects\Chest Cancer Classification End to End Project
2026/09/22 17:03:18 INFO mlflow.utils.environment: Detected uv project at d:\Projects\1-Practice and Learning\0 - COMPLETE NEW LEARNING ML and DL\3-PROJECTS\Deep Learning Projects\Chest Cancer Classification End to End Project. Attempting to export requirements via 'uv export'.
2026/09/22 17:03:18 INFO mlflow.utils.uv_utils: Exported 255 dependencies via uv
2026/09/22 1

🏃 View run nervous-stork-940 at: https://dagshub.com/mlenthusiast0/Chest-Cancer-Classification-Deep-Learning-Project.mlflow/#/experiments/0/runs/938a675027d3492591e1f7769caba565
🧪 View experiment at: https://dagshub.com/mlenthusiast0/Chest-Cancer-Classification-Deep-Learning-Project.mlflow/#/experiments/0


**EXTRA-STEP STEP 8 : MOVE THIS ENTIRE PIPELINE TO A .PY FILE**